[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunnatillaNSH/whisper-uzbek-asr/blob/main/notebooks/whisper_uzbek_finetune_colab.ipynb)

# Whisper — O'zbek tili uchun Fine-Tuning + FastAPI + ngrok (Google Colab T4)

Bu notebook 4 bosqichdan iborat va yuqoridan pastga ketma-ket ishga tushiriladi:

1. **Environment Setup** — kutubxonalarni o'rnatish
2. **Fine-Tuning** — `openai/whisper-small` ni CSV dataset bilan o'qitish (Hugging Face `Seq2SeqTrainer`)
3. **FastAPI + ngrok** — modelni `POST /transcribe` API sifatida internetga chiqarish
4. **Integration** — cURL / Python / JavaScript mijoz kodlari

> **Runtime → Change runtime type → T4 GPU** tanlanganiga ishonch hosil qiling.

### Kutilayotgan dataset formati

```
/content/data/
├── train.csv          # ustunlar: path,sentence
└── audio/
    ├── 0001.wav
    ├── 0002.mp3
    └── ...
```

`train.csv` namunasi:

```csv
path,sentence
0001.wav,Assalomu alaykum, qanday yordam bera olaman?
0002.mp3,Buyurtmangiz ertaga yetkazib beriladi.
```

`path` ustuni `audio/` papkasiga nisbatan yoki absolyut yo'l bo'lishi mumkin. Har bir audio **30 soniyadan qisqa** bo'lishi kerak (Whisper cheklovi).


## 0. To'liq tozalash (har doim 0'dan boshlash)

Quyidagi katakcha oldingi barcha holatni (audio fayllar, saqlangan model, Hugging Face datasets keshi) **butunlay o'chiradi**. Shunda har ishga tushirish, oldingi (ehtimol yarim qolgan yoki aralash) holatdan emas, har doim **toza holatdan** boshlanadi.

> ⚠️ Bu barcha datasetlarni QAYTADAN yuklashga majbur qiladi — 20-40 daqiqa qo'shimcha vaqt ketishi mumkin, lekin natija har doim bashorat qilinadigan va ishonchli bo'ladi.

In [ ]:
# ==============================================================
# 0. TO'LIQ TOZALASH — HAR DOIM 0'DAN BOSHLASH
# ==============================================================
import shutil, os

PATHS_TO_WIPE = [
    "/content/data",
    "/content/whisper-large-v3-uz",
    "/content/whisper-large-v3-uz-v2",
    "/root/.cache/huggingface",   # HF datasets/hub keshi — datasetlar qaytadan yuklanadi
]

for p in PATHS_TO_WIPE:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f"🗑️  O'chirildi: {p}")
    else:
        print(f"   Mavjud emas, o'tkazib yuborildi: {p}")

!pip cache purge > /dev/null 2>&1
!apt-get clean > /dev/null 2>&1

print()
!df -h /content
print("\n✅ Disk to'liq tozalandi — endi 0'dan boshlaymiz.")


## 1. Environment Setup

In [ ]:
# ==============================================================
# 1. ENVIRONMENT SETUP — T4 GPU uchun kutubxonalarni o'rnatish
# ==============================================================
!nvidia-smi

# ML kutubxonalari (peft — LoRA fine-tuning uchun, large-v3'ni T4'ga sig'diradi)
!pip install -q -U "transformers>=4.46" datasets accelerate evaluate jiwer librosa soundfile "peft>=0.7" "torchao>=0.16.0"
# ^ torchao — peft ning ichki versiya tekshiruvi uchun kerak (LoRA o'zi buni ishlatmaydi,
#   lekin Colab'da oldindan o'rnatilgan eski torchao versiyasi ImportError beradi)

# API kutubxonalari
!pip install -q fastapi "uvicorn[standard]" pyngrok python-multipart nest_asyncio requests

# mp3 / m4a / webm dekodlash uchun ffmpeg (Colab'da odatda bor, ehtiyot uchun)
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

import torch, transformers, datasets, peft
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__, "| peft:", peft.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Thu Sep 17 09:52:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.1 Dataset — bir nechta ochiq o'zbekcha manbani birlashtirish

Sifat va xilma-xillikni oshirish uchun **yettita ochiq, gatingsiz** datasetni bitta pastga birlashtiramiz:

| Manba | Cheklov | Izoh |
|---|---:|---|
| [`yakhyo/mozilla-common-voice-uzbek`](https://huggingface.co/datasets/yakhyo/mozilla-common-voice-uzbek) | 8 000 | Mozilla Common Voice, tasdiqlangan (`validated`) |
| [`DavronSherbaev/uzbekvoice-filtered`](https://huggingface.co/datasets/DavronSherbaev/uzbekvoice-filtered) | 20 000 | UzbekVoice — 503k qatorlik jamoaviy loyihadan, tabiiy yig'ilgan nutq |
| [`mrmuminov/uzbek_voice`](https://huggingface.co/datasets/mrmuminov/uzbek_voice) | 30 000 | Yana bir katta jamoaviy loyiha — 861k qatorlik pool, sifat filtri (up/down vote) bilan |
| [`shunyalabs/uzbek-speech-dataset`](https://huggingface.co/datasets/shunyalabs/uzbek-speech-dataset) | hammasi (~2 943) | kichik, sifatli qo'shimcha to'plam |
| [`islomov/news_youtube_uzbek_speech_dataset`](https://huggingface.co/datasets/islomov/news_youtube_uzbek_speech_dataset) | 15 000 | YouTube yangiliklaridan, turli lahjalar |
| [`islomov/it_youtube_uzbek_speech_dataset`](https://huggingface.co/datasets/islomov/it_youtube_uzbek_speech_dataset) | 10 000 | IT mavzusidagi YouTube, ba'zi joylarda inglizcha so'zlar aralash |
| [`BoburAmirov/podcasts_tashkent_dialect_youtube_uzbek_speech_dataset`](https://huggingface.co/datasets/BoburAmirov/podcasts_tashkent_dialect_youtube_uzbek_speech_dataset) | hammasi (~14 547) | Toshkent lahjasidagi podkastlar — **tabiiy, erkin suhbat uslubi**, qo'ng'iroq tahlili uchun ayniqsa foydali |

Jami cheklov bo'yicha **~98 000 namuna** (birinchi urinishdagi 8000 tadan ~12 baravar ko'p). Har biri avtomatik yuklanadi, audio `.wav` (16 kHz) holida `/content/data/audio/` ga, transkriptlar birlashtirilgan holda `/content/data/train.csv` ga yoziladi.

**Ikkita qo'shimcha manba topildi, lekin standart o'chirilgan**:
- **FeruzaSpeech** (60 soat, yuqori sifat) — HF'da **gated**: avval [dataset sahifasida](https://huggingface.co/datasets/k2speech/FeruzaSpeech) shartlarni qabul qilish va `HF_TOKEN` kerak.
- **Uzbek Speech Corpus / USC** (105 soat) — HF'dagi standart yuklagichi ishlamayapti (WebDataset xatosi), [GitHub'dan](https://github.com/IS2AI/Uzbek_ASR) qo'lda olish talab qilinadi.

**O'z datasetingizni ishlatmoqchi bo'lsangiz** (masalan qo'ng'iroq yozuvlari), quyidagi katakchadagi kodni o'tkazib yuborib, `train.csv` va `audio/` papkasini qo'lda tayyorlang (Drive'dan nusxalash yoki zip yuklash misollari izohlarda bor).

In [ ]:
# ==============================================================
# 1.1 BIR NECHTA DATASETNI YUKLAB, BIRLASHTIRISH
# ==============================================================
import os, csv, shutil
from datasets import load_dataset, Audio
import soundfile as sf
from tqdm.auto import tqdm

DATA_DIR      = "/content/data"
AUDIO_OUT_DIR = f"{DATA_DIR}/audio"

# Oldingi ishga tushirishdan qolgan audio fayllarni tozalaymiz — aks holda
# disk asta-sekin to'lib boradi (HF kutubxonalar keshi ESA tegilmaydi,
# u qayta yuklab olishning oldini oladi, tezlashtiradi).
if os.path.exists(AUDIO_OUT_DIR):
    shutil.rmtree(AUDIO_OUT_DIR)
os.makedirs(AUDIO_OUT_DIR, exist_ok=True)

TARGET_SR = 16000   # Whisper faqat 16 kHz audio bilan ishlaydi

# Disk uchun xavfsizlik chegarasi: bo'sh joy shu miqdorgacha tushsa, QOLGAN
# manbalarni yuklashni to'xtatamiz va HOZIRGACHA yig'ilgan namunalar bilan
# davom etamiz — diskning to'lib, xato bilan yiqilib qolishidan ko'ra yaxshi.
MIN_FREE_DISK_GB = 20

def free_disk_gb():
    return shutil.disk_usage("/content").free / (1024 ** 3)

# --- Manbalar: (HF dataset id, split, matn ustuni nomzodlari, cheklov, fayl prefiksi) ---
# cheklov=None bo'lsa — shu manbadagi barcha namunalar olinadi
SOURCES = [
    ("yakhyo/mozilla-common-voice-uzbek",                                    "validated", ["sentence", "text"],                        8000,  "cv"),
    ("DavronSherbaev/uzbekvoice-filtered",                                    "train",     ["text", "sentence"],                        20000, "uv"),
    ("mrmuminov/uzbek_voice",                                                 "train",     ["original_sentence", "text", "sentence"],   30000, "mv"),
    ("shunyalabs/uzbek-speech-dataset",                                       "train",     ["transcript", "text", "transcription", "sentence"], None,  "sh"),
    ("islomov/news_youtube_uzbek_speech_dataset",                             "train",     ["text", "sentence"],                        15000, "yt"),
    ("islomov/it_youtube_uzbek_speech_dataset",                               "train",     ["text", "sentence"],                        10000, "ityt"),
    ("BoburAmirov/podcasts_tashkent_dialect_youtube_uzbek_speech_dataset",    "train",     ["text", "sentence"],                        None,  "pod"),

    # --- Ixtiyoriy, standart o'chirilgan (sabab yuqoridagi izohda) ---
    # FeruzaSpeech: gated — avval https://huggingface.co/datasets/k2speech/FeruzaSpeech da
    # shartlarni qabul qiling, Colab Secrets'ga HF_TOKEN qo'shing, keyin quyidagi qatorni oching
    # (ustun nomlari taxminiy — birinchi ishga tushirishda tekshiring):
    # ("k2speech/FeruzaSpeech", "train", ["text_latin", "sentence", "text"], None, "fz"),
]

def get_text(example, candidates):
    for col in candidates:
        val = example.get(col)
        if val:
            return str(val).strip()
    return ""

def load_and_filter(hf_id, split, text_cols, max_samples):
    ds = load_dataset(hf_id, split=split)
    print(f"  Ustunlar: {ds.column_names}")

    # Audio ustunini NOMI bo'yicha emas, TURI (Audio feature) bo'yicha topamiz —
    # ba'zi datasetlarda bu ustun "audio" deb emas, boshqacha nomlangan bo'lishi mumkin
    # (masalan uzbekvoice-filtered'da audio aslida "path" ustunida saqlangan).
    audio_col = next((name for name, feat in ds.features.items() if isinstance(feat, Audio)), None)
    if audio_col is None:
        raise ValueError(f"Audio ustuni topilmadi. Mavjud ustunlar: {ds.column_names}")
    if audio_col != "audio":
        ds = ds.rename_column(audio_col, "audio")
        print(f"  Audio ustuni: '{audio_col}' → 'audio' deb qayta nomlandi")
    ds = ds.cast_column("audio", Audio(sampling_rate=TARGET_SR))

    # Matni bo'sh bo'lgan qatorlarni OLDINDAN chiqarib tashlaymiz — aks holda
    # keyingi cheklov (select) so'ralgan miqdorni bermay, "yozish" bosqichida
    # ko'p qator sababsiz jim tashlab yuborilib qolardi (aynan shu 1-marta
    # ishga tushirishda uv/sh manbalarida sodir bo'lgan muammo).
    before = len(ds)
    ds = ds.filter(lambda x: any(x.get(c) for c in text_cols))
    if len(ds) < before:
        print(f"  Bo'sh matnli qatorlar chiqarildi: {before} → {len(ds)}")

    # Sifat filtri (up/down vote) — ustun nomi manbadan manbaga farq qilishi mumkin
    upvote_col   = next((c for c in ("up_votes", "upvotes", "up_vote") if c in ds.column_names), None)
    downvote_col = next((c for c in ("down_votes", "downvotes", "down_vote") if c in ds.column_names), None)
    if upvote_col and downvote_col:
        before = len(ds)
        ds = ds.filter(lambda x: x[upvote_col] >= 1 and x[downvote_col] == 0)
        print(f"  Sifat filtri ({upvote_col}/{downvote_col}): {before} → {len(ds)} qator")

    ds = ds.shuffle(seed=42)
    if max_samples is not None:
        ds = ds.select(range(min(max_samples, len(ds))))
    return ds

rows = []
disk_limit_hit = False
for hf_id, split, text_cols, max_samples, prefix in SOURCES:
    free_gb = free_disk_gb()
    if free_gb < MIN_FREE_DISK_GB:
        print(f"\n🛑 Bo'sh disk joyi {free_gb:.1f} GB ga tushdi (chegara: {MIN_FREE_DISK_GB} GB).")
        print(f"   Qolgan manbalar o'tkazib yuborilib, mavjud {len(rows)} namuna bilan davom etamiz.")
        disk_limit_hit = True
        break

    print(f"\n📥 {hf_id} ({split}) yuklanmoqda... (bo'sh joy: {free_gb:.1f} GB)")
    try:
        ds = load_and_filter(hf_id, split, text_cols, max_samples)
    except Exception as e:
        print(f"  ⚠️  O'tkazib yuborildi (yuklab bo'lmadi): {e}")
        continue
    print(f"  Tanlangan namunalar: {len(ds)}")

    written, skipped = 0, 0
    for i, ex in enumerate(tqdm(ds, desc=f"{prefix}: audio yozilmoqda")):
        # Har 2000 qatorda diskni tekshiramiz — bitta katta manba o'zi
        # chegaradan pastga tushirib yubormasligi uchun
        if i % 2000 == 0 and free_disk_gb() < MIN_FREE_DISK_GB:
            print(f"\n🛑 Bo'sh disk joyi {MIN_FREE_DISK_GB} GB chegarasiga yetdi, '{prefix}' manbasi shu yerda to'xtatildi.")
            disk_limit_hit = True
            break
        text = get_text(ex, text_cols)
        audio = ex.get("audio")
        if not text or not audio:
            skipped += 1
            continue
        fname = f"{prefix}_{i:06d}.wav"
        sf.write(os.path.join(AUDIO_OUT_DIR, fname), audio["array"], audio["sampling_rate"])
        rows.append({"path": fname, "sentence": text})
        written += 1
    print(f"  Yozildi: {written} | O'tkazib yuborildi: {skipped}")
    del ds
    if disk_limit_hit:
        break

with open(f"{DATA_DIR}/train.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["path", "sentence"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\n✅ Jami tayyor: {len(rows)} namuna ({len(SOURCES)} manbadan) → {DATA_DIR}/train.csv va {AUDIO_OUT_DIR}/")

# --- Muqobil: o'z CSV/audio to'plamingizni Drive'dan yoki zip'dan olish ---
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r "/content/drive/MyDrive/uzbek_asr/." {DATA_DIR}/
#
# !unzip -q /content/drive/MyDrive/uzbek_asr.zip -d {DATA_DIR}


## 2. Fine-Tuning Pipeline

### Nega LoRA (PEFT)?

`whisper-large-v3` — eng kuchli Whisper modeli (1.5 milliard parametr). Uni **to'liq** fine-tune qilish T4'ning 15 GB xotirasiga sig'maydi (Adam optimizatori o'zi ~25 GB talab qiladi).

Yechim — **LoRA**: asosiy model fp16'da muzlatilgan holda yuklanadi, faqat attention qatlamlaridagi kichik adapter matritsalari (~0.3% parametrlar) o'qitiladi. Sifat to'liq fine-tuning'ga juda yaqin, lekin T4'da bemalol sig'adi. Trening tugagach adapterlar asosiy modelga **birlashtiriladi** (`merge_and_unload`) — natija oddiy Whisper modeli, FastAPI serveri hech qanday qo'shimcha kutubxonasiz uni to'g'ridan-to'g'ri yuklaydi.

> ⚠️ **Vaqt haqida**: `large-v3` `small`ga nisbatan 5-8 baravar sekinroq. T4'da 1500 qadam taxminan **6-10 soat** davom etadi. Colab bepul sessiyasi ~12 soatdan keyin uziladi — uzoq faolsizlikda ham uzilishi mumkin, shuning uchun tab'ni ochiq qoldiring va vaqti-vaqti bilan sahifaga qayting. Uzilsa, 2.7-katakchada `trainer.train(resume_from_checkpoint=True)` bilan davom ettiring.

In [ ]:
# ==============================================================
# 2.0 GLOBAL SOZLAMALAR — barcha keyingi katakchalar shu qiymatlardan foydalanadi
# ==============================================================
import os, gc, torch

CSV_PATH   = "/content/data/train.csv"       # path,sentence ustunli CSV
AUDIO_DIR  = "/content/data/audio"           # nisbiy path'lar shu papkaga nisbatan

MODEL_NAME = "openai/whisper-large-v3"       # eng kuchli Whisper modeli (1.5 mlrd parametr)
OUTPUT_DIR = "/content/whisper-large-v3-uz"  # fine-tuned (merge qilingan) model shu yerga saqlanadi
# Kelajakda YANA HAM ko'proq ma'lumot bilan davom ettirib o'qitmoqchi bo'lsangiz,
# MODEL_NAME'ni shu OUTPUT_DIR (yoki Drive nusxasi) ga o'zgartiring va OUTPUT_DIR'ga
# boshqa nom bering — hozircha hammasi bitta katta bosqichda o'qitiladi.

LANGUAGE = "uzbek"
TASK     = "transcribe"

SAMPLING_RATE = 16000       # Whisper faqat 16 kHz audio bilan ishlaydi
MAX_AUDIO_SEC = 30.0        # Whisper bir oynada 30 soniya qabul qiladi
TEST_SIZE       = 0.1       # validatsiya ulushi (train/test bo'linishda)
EVAL_MAX_SAMPLES = 500      # lekin trening DAVOMIDAGI tez-tez baholashlar shu miqdorga cheklanadi —
                             # aks holda ~4-5 ming namunali eval to'plamini har necha yuz qadamda
                             # to'liq generate() qilish soatlab vaqt yeyishi mumkin (generatsiya sekin)

# Trening giperparametrlari — A100 (40/80 GB) + LoRA uchun sozlangan.
# Kichikroq GPU (T4, 15 GB) ishlatsangiz: BATCH_SIZE=2, GRAD_ACCUM=8 ga qaytaring.
# max_steps dataset hajmiga QARAMAYDI (epoch emas, qadam soni bilan boshqariladi) —
# ~98 000 namunali birlashtirilgan dataset uchun 6000 qadam ≈ 1 epoch atrofida.
# Colab compute unit budjetini tejash uchun 15000 → 10000 → 6000 ga tushirildi.
MAX_STEPS   = 6000
EVAL_STEPS  = 500
BATCH_SIZE  = 8              # A100'da xotira yetarli — T4'da 2 ga tushiring
GRAD_ACCUM  = 2               # effektiv batch = 8 * 2 = 16 (T4 uchun: 2 * 8 = 16, o'zgarmaydi)
LR          = 1e-4          # LoRA uchun to'liq fine-tuning'dan yuqoriroq LR tavsiya etiladi
WARMUP_STEPS = 500           # uzunroq trening uchun uzunroq isinish (avvalgi 100 o'rniga)

# --- LoRA sozlamalari — katta modelni cheklangan GPU'da o'qitish imkonini beradi ---
LORA_R              = 32                      # adapter rangi — kattaroq = ko'proq sig'im, ko'proq xotira
LORA_ALPHA          = 64                      # odatda 2*r
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]    # attention'dagi query/value proyeksiyalari

torch.manual_seed(42)
print("Sozlamalar tayyor. Model:", MODEL_NAME, "| usul: LoRA (PEFT)")


In [ ]:
# ==============================================================
# 2.1 CSV DATASETNI YUKLASH VA `datasets` FORMATIGA O'TKAZISH
# ==============================================================
import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_csv(CSV_PATH)
assert {"path", "sentence"} <= set(df.columns), "CSV'da 'path' va 'sentence' ustunlari bo'lishi shart"

# Nisbiy yo'llarni AUDIO_DIR bilan birlashtiramiz
df["path"] = df["path"].astype(str).apply(
    lambda p: p if os.path.isabs(p) else os.path.join(AUDIO_DIR, p)
)

# Tozalash: mavjud bo'lmagan fayllar va bo'sh matnlarni olib tashlaymiz
before = len(df)
df = df[df["path"].apply(os.path.exists)]
df = df.dropna(subset=["sentence"])
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"] != ""].reset_index(drop=True)
print(f"Jami qatorlar: {before} → yaroqli: {len(df)}")

# pandas → datasets, so'ng train/test bo'linish
full_ds = Dataset.from_pandas(df[["path", "sentence"]], preserve_index=False)
split   = full_ds.train_test_split(test_size=TEST_SIZE, seed=42)

# Eval to'plamini EVAL_MAX_SAMPLES bilan cheklaymiz — trening davomidagi tez-tez
# baholashlar (predict_with_generate=True) katta eval to'plamida juda sekin bo'ladi,
# chunki har bir namuna uchun matn generatsiya qilinadi (autoregressiv, sekin).
eval_split = split["test"]
if len(eval_split) > EVAL_MAX_SAMPLES:
    eval_split = eval_split.shuffle(seed=42).select(range(EVAL_MAX_SAMPLES))

dataset = DatasetDict(train=split["train"], test=eval_split)
print(dataset)
print(f"(eval to'plami {EVAL_MAX_SAMPLES} namunaga cheklandi, to'liq bo'lsa {len(split['test'])} bo'lar edi)")


In [ ]:
# ==============================================================
# 2.2 WHISPER PROCESSOR VA MODELNI O'ZBEK TILI UCHUN YUKLASH (+ LoRA)
# ==============================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

# Processor = FeatureExtractor (audio → log-mel) + Tokenizer (matn → token)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# Asosiy modelni fp32'da yuklaymiz — aralash aniqlik (mixed precision, fp16)
# training_args'dagi fp16=True orqali avtomatik boshqariladi (autocast), bu
# HAM trening, HAM baholash/generate bosqichida bir xil ishlaydi. Modelni
# to'g'ridan-to'g'ri fp16'da yuklab, ustiga fp16=True qo'shsak, generate()
# paytida "Input type (float) and bias type (Half) should be the same"
# degan xatolikka olib keladi — shuning uchun bu yerda dtype ko'rsatmaymiz.
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

# Generatsiya vaqtida til va vazifani majburiy belgilaymiz (eski forced_decoder_ids o'rniga)
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

# gradient checkpointing bilan mos bo'lishi uchun cache'ni o'chiramiz
model.config.use_cache = False

# Audio kiruvchi qatlam gradientni uzatishi uchun kerak (muzlatilgan model +
# gradient checkpointing kombinatsiyasida majburiy — rasmiy PEFT+Whisper
# tavsiyasi). LoRA bilan o'rash'dan OLDIN, asl modulga ro'yxatdan o'tkazamiz.
def _make_inputs_require_grad(module, input, output):
    output.requires_grad_(True)
model.model.encoder.conv1.register_forward_hook(_make_inputs_require_grad)

# --- LoRA bilan o'rab olamiz: asosiy model muzlatiladi, faqat adapterlar o'qitiladi ---
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # masalan: ~5M / ~1550M (~0.3%) o'qitiladi


In [ ]:
# ==============================================================
# 2.3 FEATURE EXTRACTION + TOKENIZATION (prepare_dataset)
# ==============================================================
import librosa
import numpy as np

MAX_LABEL_LEN = model.config.max_target_positions  # Whisper dekoderi maksimal 448 token

def prepare_dataset(batch):
    """Bitta namunani: audio faylni 16kHz'ga o'qiydi → log-mel xususiyatlar,
    matnni → token id'lar ko'rinishiga o'tkazadi."""
    # librosa wav/mp3/flac/ogg'ni o'qiydi va avtomatik 16 kHz mono'ga resample qiladi
    audio, _ = librosa.load(batch["path"], sr=SAMPLING_RATE, mono=True)

    # Audio → log-mel spektrogram (80 x 3000)
    batch["input_features"] = processor.feature_extractor(
        audio, sampling_rate=SAMPLING_RATE
    ).input_features[0]

    # Filtrlash uchun davomiylik va token uzunligi
    batch["input_length"] = len(audio) / SAMPLING_RATE

    # Matn → token id'lar (til/vazifa prefikslari avtomatik qo'shiladi)
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    batch["labels_length"] = len(batch["labels"])
    return batch

# num_proc BERILMAYDI (na 1, na undan ko'p): model allaqachon GPU'ga
# yuklangan (2.2-bo'lim), datasets kutubxonasi num_proc>=1 bo'lganda ham
# ba'zan alohida process (fork) yaratadi — bu CUDA konteksti bilan
# to'qnashib abadiy muzlab qolishiga (hang) olib kelishi mumkin.
# num_proc ko'rsatilmasa, joriy jarayonning o'zida, process yaratmasdan
# ishlaydi — bu eng ishonchli variant.
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    desc="Audio va matnni tayyorlash",
)

# 30 soniyadan uzun audio va 448 tokendan uzun matnlarni chiqarib tashlaymiz
def is_valid(input_length, labels_length):
    return input_length <= MAX_AUDIO_SEC and labels_length <= MAX_LABEL_LEN

dataset = dataset.filter(is_valid, input_columns=["input_length", "labels_length"])
dataset = dataset.remove_columns(["input_length", "labels_length"])
print(dataset)


In [ ]:
# ==============================================================
# 2.4 DATA COLLATOR — batch ichida audio va labellarni padding qiladi
# ==============================================================
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Audio xususiyatlari allaqachon 3000 kadrga to'ldirilgan → faqat tensorga o'tkazamiz
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Labellarni batch ichidagi eng uzun matn uzunligiga qadar padding qilamiz
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Padding tokenlarni -100 bilan almashtiramiz → loss hisobida e'tiborga olinmaydi
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Tokenizer boshiga <|startoftranscript|> qo'shgan bo'lsa, olib tashlaymiz
        # (model uni generatsiya vaqtida o'zi qo'shadi)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print("Data collator tayyor")


In [ ]:
# ==============================================================
# 2.5 METRIKA — WER (Word Error Rate), qancha kichik bo'lsa shuncha yaxshi
# ==============================================================
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids

    # Ba'zi versiyalarda predictions tuple bo'lib keladi
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    # -100 ni pad tokenga qaytaramiz, aks holda decode xato beradi
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


In [ ]:
# ==============================================================
# 2.6 SEQ2SEQ TRAINING ARGUMENTS — 2.0-bo'limdagi BATCH_SIZE/GRAD_ACCUM'dan foydalanadi
# ==============================================================
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batch va xotira
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,        # effektiv batch 16
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_checkpointing=True,                   # VRAM'ni ~40% tejaydi
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=True,                                     # aralash aniqlik (autocast)

    # Optimizatsiya
    learning_rate=LR,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    lr_scheduler_type="linear",

    # Baholash va saqlash — DIQQAT: predict_with_generate ISHLATILMAYDI.
    # PEFT (LoRA) + Seq2SeqTrainer'ning ichki generate() chaqiruvi ma'lum,
    # hujjatlashtirilgan nomuvofiqlikka ega — generate() autocast'dan
    # tashqarida ishga tushib, "Input type (float) and bias type (Half)
    # should be the same" xatosini beradi (PEFT+Whisper community'da
    # tanilgan muammo). Shuning uchun trening davomida faqat LOSS orqali
    # baholaymiz (tez va barqaror), WER'ni esa 2.8-bo'limda, trening
    # tugagach, generate()'ni QO'LDA (Trainer'dan tashqarida) chaqirib
    # bir marta hisoblaymiz.
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,                            # disk to'lib qolmasligi uchun
    load_best_model_at_end=True,
    metric_for_best_model="loss",                  # eval_wer o'rniga eval_loss
    greater_is_better=False,

    # PEFT (LoRA) modellari uchun zarur: Trainer forward signaturasidan
    # "labels" ustunini avtomatik topa olmaydi — qo'lda ko'rsatamiz
    label_names=["labels"],
    # PeftModel'ning forward signaturasi generic (**kwargs) bo'lgani uchun
    # kerakli ustunlarni avtomatik olib tashlashni o'chiramiz
    remove_unused_columns=False,

    # Logging
    logging_steps=25,
    report_to="none",                              # wandb/tensorboard'siz

    dataloader_num_workers=2,
    push_to_hub=False,
)
print("Training arguments tayyor")


In [ ]:
# ==============================================================
# 2.7 SEQ2SEQ TRAINER — modelni o'qitish
# ==============================================================
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    # compute_metrics BERILMAYDI — predict_with_generate=False bo'lgani uchun
    # trening davomidagi baholash faqat loss orqali, WER 2.8-bo'limda qo'lda
    # hisoblanadi (yuqoridagi izohga qarang)
    processing_class=processor,      # transformers >= 4.46 (eski nomi: tokenizer=)
)

# Colab uzilib qolsa: trainer.train(resume_from_checkpoint=True) bilan davom ettirish mumkin
train_result = trainer.train()
print(train_result)


In [ ]:
# ==============================================================
# 2.8 LoRA ADAPTERLARNI BIRLASHTIRISH, YAKUNIY WER VA SAQLASH
# ==============================================================
# LoRA adapterlarni asosiy modelga birlashtiramiz (merge) — natija oddiy
# WhisperForConditionalGeneration bo'ladi. FastAPI serveri (app.py) buni
# PEFT kutubxonasisiz, o'zgarishsiz to'g'ridan-to'g'ri yuklay oladi.
merged_model = trainer.model.merge_and_unload()
merged_model = merged_model.half()   # diskda ~2x kichikroq, inference tezroq
merged_model.eval()

# Yakuniy WER'ni QO'LDA hisoblaymiz (Trainer'ning predict_with_generate'idan
# TASHQARIDA) — sabab yuqoridagi 2.6-bo'limdagi izohda. Endi model va kirish
# ikkalasi ham fp16, dtype mos keladi, xatolik chiqmaydi.
print("Yakuniy WER hisoblanmoqda...")
all_preds, all_refs = [], []
with torch.no_grad():
    for i in range(0, len(dataset["test"]), BATCH_SIZE):
        batch = dataset["test"][i : i + BATCH_SIZE]
        feats = torch.tensor(batch["input_features"]).to(merged_model.device).half()
        pred_ids = merged_model.generate(feats, language=LANGUAGE, task=TASK, max_new_tokens=225)
        all_preds.extend(processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True))
        all_refs.extend(processor.tokenizer.decode(l, skip_special_tokens=True) for l in batch["labels"])

final_wer = 100 * wer_metric.compute(predictions=all_preds, references=all_refs)
print("Yakuniy WER (%):", round(final_wer, 2))

merged_model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print("Saqlandi:", os.listdir(OUTPUT_DIR))

# --- Ixtiyoriy: Drive'ga nusxalash (Colab sessiyasi tugasa ham saqlanib qoladi) ---
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r {OUTPUT_DIR} /content/drive/MyDrive/whisper-large-v3-uz


In [ ]:
# ==============================================================
# 2.9 TEZKOR SINOV — test to'plamidan bitta namuna
# ==============================================================
sample = dataset["test"][0]
input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(merged_model.device).half()
with torch.no_grad():
    pred_ids = merged_model.generate(input_features, language=LANGUAGE, task=TASK, max_new_tokens=225)

print("Asl matn :", processor.tokenizer.decode(sample["labels"], skip_special_tokens=True))
print("Bashorat :", processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True))

# GPU xotirasini bo'shatamiz — API serverda model qayta yuklanadi
del trainer, model, merged_model
gc.collect()
torch.cuda.empty_cache()
print("GPU bo'sh xotira (GB):", round(torch.cuda.mem_get_info()[0] / 1e9, 2))


## 3. FastAPI Server + ngrok

Avval `app.py` fayli yoziladi (uni Colab'dan tashqarida ham `uvicorn app:app` bilan ishlatish mumkin), keyin notebook ichida background'da ishga tushiriladi.

In [ ]:
%%writefile app.py
# ==============================================================
# 3.1 FASTAPI ILOVASI — fine-tuned Whisper modelini API sifatida xizmat qilish
# ==============================================================
import os
import time
import shutil
import tempfile
import threading
import subprocess

import torch
import librosa
from fastapi import FastAPI, File, UploadFile, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from transformers import pipeline

# ---------------- Sozlamalar ----------------
MODEL_DIR     = os.environ.get("MODEL_DIR", "/content/whisper-small-uz")
LANGUAGE      = os.environ.get("ASR_LANGUAGE", "uzbek")
SAMPLING_RATE = 16000
MAX_FILE_MB   = 50
ALLOWED_EXT   = {".wav", ".mp3", ".m4a", ".ogg", ".oga", ".flac", ".webm", ".opus", ".aac"}

# ---------------- Modelni bir marta yuklash ----------------
DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"[app] Model yuklanmoqda: {MODEL_DIR} (device={DEVICE}, dtype={DTYPE})")
asr = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_DIR,
    device=DEVICE,
    torch_dtype=DTYPE,
    chunk_length_s=30,      # 30 soniyadan uzun audiolarni bo'laklab transkripsiya qiladi
)
print("[app] Model tayyor")

# GPU'da bir vaqtda faqat bitta inference bo'lishi uchun qulf
_infer_lock = threading.Lock()

# ---------------- FastAPI ----------------
app = FastAPI(
    title="Uzbek Whisper ASR API",
    description="Fine-tuned Whisper modeli orqali o'zbek tilidagi audioni matnga o'girish",
    version="1.0.0",
)

# Boshqa domenlardan (frontend, mobil ilova) so'rov yuborishga ruxsat
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Barcha xatolarni bir xil JSON formatda qaytaramiz: {"status": "error", "message": "..."}
@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(status_code=exc.status_code, content={"status": "error", "message": exc.detail})

@app.exception_handler(Exception)
async def generic_exception_handler(request: Request, exc: Exception):
    return JSONResponse(status_code=500, content={"status": "error", "message": str(exc)})


def load_audio(path: str):
    """Audio faylni 16 kHz mono numpy massiviga o'qiydi.
    librosa o'qiy olmasa (masalan webm/m4a), ffmpeg orqali wav'ga o'giradi."""
    try:
        audio, _ = librosa.load(path, sr=SAMPLING_RATE, mono=True)
        return audio
    except Exception:
        wav_path = path + ".16k.wav"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", path,
             "-ac", "1", "-ar", str(SAMPLING_RATE), wav_path],
            check=True,
        )
        try:
            audio, _ = librosa.load(wav_path, sr=SAMPLING_RATE, mono=True)
            return audio
        finally:
            if os.path.exists(wav_path):
                os.remove(wav_path)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_DIR,
        "device": "cuda" if DEVICE == 0 else "cpu",
        "language": LANGUAGE,
    }


# `async def` emas, `def` — FastAPI uni threadpool'da ishga tushiradi va
# og'ir GPU inference event loop'ni bloklamaydi.
@app.post("/transcribe")
def transcribe(file: UploadFile = File(...)):
    filename = file.filename or "audio"
    ext = os.path.splitext(filename)[1].lower()
    if ext not in ALLOWED_EXT:
        raise HTTPException(
            status_code=400,
            detail=f"Qo'llab-quvvatlanmaydigan format '{ext}'. Ruxsat etilgan: {sorted(ALLOWED_EXT)}",
        )

    tmp_path = None
    try:
        # 1) Faylni vaqtinchalik saqlaymiz
        with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
            shutil.copyfileobj(file.file, tmp)
            tmp_path = tmp.name

        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
        if size_mb > MAX_FILE_MB:
            raise HTTPException(status_code=413, detail=f"Fayl juda katta ({size_mb:.1f} MB). Limit: {MAX_FILE_MB} MB")

        # 2) Audio → 16 kHz massiv
        audio = load_audio(tmp_path)
        if audio is None or len(audio) == 0:
            raise HTTPException(status_code=400, detail="Audio bo'sh yoki o'qib bo'lmadi")

        # 3) Transkripsiya (til va vazifa majburiy belgilanadi)
        t0 = time.time()
        with _infer_lock:
            result = asr(
                {"raw": audio, "sampling_rate": SAMPLING_RATE},
                generate_kwargs={"language": LANGUAGE, "task": "transcribe"},
            )
        elapsed = time.time() - t0

        # 4) JSON javob
        return {
            "status": "success",
            "text": result["text"].strip(),
            "filename": filename,
            "duration_sec": round(len(audio) / SAMPLING_RATE, 2),
            "processing_time_sec": round(elapsed, 2),
        }
    finally:
        # 5) Vaqtinchalik faylni tozalaymiz
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


In [ ]:
# ==============================================================
# 3.2 UVICORN'NI BACKGROUND'DA ISHGA TUSHIRISH + NGROK ORQALI PUBLIC URL
# ==============================================================
import os, sys, time, threading, requests
import nest_asyncio, uvicorn
from pyngrok import ngrok

nest_asyncio.apply()  # Colab'ning o'z event loop'i bilan to'qnashmaslik uchun

# app.py'ga model yo'lini beramiz va import qilamiz (model shu jarayonda GPU'ga yuklanadi)
os.environ["MODEL_DIR"] = OUTPUT_DIR
if "app" in sys.modules:
    del sys.modules["app"]          # katakcha qayta ishga tushsa, eskirgan modulni tozalaymiz
import app as app_module

# --- Uvicorn serverini alohida thread'da ishga tushiramiz ---
PORT = 8000
config = uvicorn.Config(app_module.app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Server tayyor bo'lguncha kutamiz
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            print("✅ Lokal server tayyor:", f"http://127.0.0.1:{PORT}")
            break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Server 60 soniyada ishga tushmadi — yuqoridagi loglarni tekshiring")

# --- ngrok tunneli ---
# Token: https://dashboard.ngrok.com/get-started/your-authtoken
# Tavsiya: Colab → chap panel 🔑 Secrets → NGROK_AUTH_TOKEN nomi bilan saqlang
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = "BU_YERGA_NGROK_TOKENINGIZNI_QO'YING"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()                                  # eski tunnellarni yopamiz
tunnel = ngrok.connect(PORT, "http")
PUBLIC_URL = tunnel.public_url

print("\n" + "=" * 60)
print("🌍 PUBLIC URL     :", PUBLIC_URL)
print("📄 Swagger docs   :", PUBLIC_URL + "/docs")
print("🎤 Transcribe     :", PUBLIC_URL + "/transcribe  (POST, multipart/form-data, field=file)")
print("=" * 60)


In [ ]:
# ==============================================================
# 3.3 API'NI COLAB ICHIDAN SINAB KO'RISH
# ==============================================================
import requests

TEST_AUDIO = df["path"].iloc[0]   # datasetdan birinchi fayl; o'zingiznikini qo'ying

with open(TEST_AUDIO, "rb") as f:
    r = requests.post(
        f"{PUBLIC_URL}/transcribe",
        files={"file": (os.path.basename(TEST_AUDIO), f, "audio/wav")},
        timeout=120,
    )
print(r.status_code, r.json())


In [ ]:
# ==============================================================
# 3.4 SERVERNI TO'XTATISH (kerak bo'lganda)
# ==============================================================
# server.should_exit = True
# ngrok.disconnect(PUBLIC_URL)
# ngrok.kill()
# print("Server va tunnel to'xtatildi")


## 4. API Documentation & Integration

### Endpoint

| Method | Path | Body | Javob |
|---|---|---|---|
| `GET` | `/health` | — | `{"status":"ok","model":...,"device":"cuda"}` |
| `POST` | `/transcribe` | `multipart/form-data`, field nomi **`file`** | `{"status":"success","text":"...","duration_sec":5.2,"processing_time_sec":0.8}` |

Xato holatida: `{"status":"error","message":"..."}` (HTTP 400 / 413 / 500).

Qo'llab-quvvatlanadigan formatlar: wav, mp3, m4a, ogg, flac, webm, opus, aac. Limit: 50 MB.

> `PUBLIC_URL` ni yuqoridagi 3.2 katakcha chiqargan ngrok manzili bilan almashtiring (masalan `https://a1b2-34-56-78-90.ngrok-free.app`). Bepul ngrok'da URL har qayta ishga tushirishda o'zgaradi.

### cURL

```bash
curl -X POST "https://YOUR-NGROK-URL.ngrok-free.app/transcribe" \
  -H "ngrok-skip-browser-warning: true" \
  -F "file=@/path/to/audio.wav"
```

### Python (`requests`)

```python
import requests

API_URL = "https://YOUR-NGROK-URL.ngrok-free.app"

def transcribe(audio_path: str) -> str:
    with open(audio_path, "rb") as f:
        resp = requests.post(
            f"{API_URL}/transcribe",
            files={"file": (audio_path.split("/")[-1], f)},
            headers={"ngrok-skip-browser-warning": "true"},
            timeout=300,
        )
    data = resp.json()
    if data.get("status") != "success":
        raise RuntimeError(f"API xatosi: {data.get('message')}")
    return data["text"]

print(transcribe("audio.wav"))
```

### JavaScript — brauzer (`fetch` + `<input type="file">`)

```html
<input type="file" id="audio" accept="audio/*" />
<button onclick="send()">Transkripsiya</button>
<pre id="out"></pre>

<script>
const API_URL = "https://YOUR-NGROK-URL.ngrok-free.app";

async function send() {
  const file = document.getElementById("audio").files[0];
  if (!file) return alert("Audio fayl tanlang");

  const form = new FormData();
  form.append("file", file, file.name);

  const res = await fetch(`${API_URL}/transcribe`, {
    method: "POST",
    headers: { "ngrok-skip-browser-warning": "true" },
    body: form,
  });
  const data = await res.json();
  document.getElementById("out").textContent =
    data.status === "success" ? data.text : "Xato: " + data.message;
}
</script>
```

### JavaScript — Node.js 18+ / Cloudflare Workers (`fetch`)

```javascript
// Node.js 18+ da global fetch/FormData/Blob mavjud
import { readFile } from "node:fs/promises";

const API_URL = "https://YOUR-NGROK-URL.ngrok-free.app";

export async function transcribe(filePath) {
  const bytes = await readFile(filePath);
  const form = new FormData();
  form.append("file", new Blob([bytes], { type: "audio/wav" }), "audio.wav");

  const res = await fetch(`${API_URL}/transcribe`, {
    method: "POST",
    headers: { "ngrok-skip-browser-warning": "true" },
    body: form,
  });
  const data = await res.json();
  if (data.status !== "success") throw new Error(data.message);
  return data.text;
}

// Cloudflare Worker ichida (R2 yoki so'rovdan kelgan ArrayBuffer bilan):
// const form = new FormData();
// form.append("file", new Blob([arrayBuffer], { type: "audio/mp3" }), "call.mp3");
// const res = await fetch(`${API_URL}/transcribe`, { method: "POST", body: form });
```

### Eslatmalar

- **`ngrok-skip-browser-warning` header** — bepul ngrok'da brauzer ogohlantirish sahifasini o'tkazib yuboradi. cURL/Python'da majburiy emas, brauzerdan `fetch` uchun kerak.
- **Bir vaqtda so'rovlar** — server GPU'da so'rovlarni navbat bilan bajaradi (qulf orqali). Yuqori yuklama uchun `whisper-small` + `chunk_length_s` yetarli, aks holda batch inference qo'shing.
- **`whisper-medium`** uchun 2.0 katakchada `MODEL_NAME`, `BATCH_SIZE=4`, `GRAD_ACCUM=4` qiling; T4'da ~2.5x sekinroq o'qitiladi.
- **Colab uzilsa** — 2.7 katakchada `trainer.train(resume_from_checkpoint=True)`; model Drive'ga saqlangan bo'lsa 2.0 da `OUTPUT_DIR` ni Drive yo'liga o'zgartirib, 2.1–2.9 ni o'tkazib yuborib to'g'ridan-to'g'ri 3-bo'limni ishga tushirish mumkin.
